# Layer V ET Identity Gene Analysis — DOWN-regulated (Suppressed) Genes
Identifies genes that are **lower in Layer V ET** than other cell types through stepwise comparison.
These are identity-defining suppressed genes: the epigenetic discordance notebook expects them
as `FORCED_SUPPRESSION` candidates.

Stepwise logic (mirror of the UP notebook, but direction flipped):
- **Non-Neuron vs Layer V ET** → genes DOWN in L5 ET vs non-neurons (neuron-suppressed)
- **Inhibitory-Neuron vs Layer V ET** → above AND DOWN vs inhibitory (excitatory-neuron-suppressed)
- **Upper Layer vs Layer V ET** → above AND DOWN vs upper layer (deep-layer-suppressed)
- **Other Deep Layer vs Layer V ET** → above AND DOWN vs ALL other deep layer (L5 ET-specific suppression)

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path

sc.settings.verbosity = 1

/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: 

## 1. Load Data

In [2]:
# ── EDIT THESE PATHS ─────────────────────────────────────────────────────────
H5AD_PATH = "/home/nakagawa/datasets/h5ad/10X_cells_v3_AIBS.h5ad"
OUT_DIR   = Path("/home/nakagawa/datasets/LayerV_ET_results_rnaseq")
# ─────────────────────────────────────────────────────────────────────────────

OUT_DIR.mkdir(parents=True, exist_ok=True)
adata = sc.read_h5ad(H5AD_PATH)
print(adata)

AnnData object with n_obs × n_vars = 71183 × 30198
    obs: 'aggr_num', 'umi.counts', 'gene.counts', 'library_id', 'tube_barcode', 'Seq_batch', 'Region', 'Lib_type', 'donor_id', 'Amp_Name', 'Amp_Date', 'Amp_PCR_cyles', 'Lib_Date', 'Replicate_Lib', 'Lib_PCR_cycles', 'Lib_PassFail', 'Cell_Capture', 'Lib_Cells', 'Mean_Reads_perCell', 'Median_Genes_perCell', 'Median_UMI_perCell', 'Saturation', 'Live_percent', 'Total_Cells', 'Live_Cells', 'exp_component_name', 'mapped_reads', 'unmapped_reads', 'nonconf_mapped_reads', 'total.reads', 'doublet.score', 'row', 'BICCN_cluster_id', 'QC', 'BICCN_cluster_label', 'BICCN_subclass_label', 'BICCN_class_label', 'size', 'temp_class_label', 'BICCN_ontology_term_id', 'assay_ontology_term_id', 'disease_ontology_term_id', 'tissue_ontology_term_id', 'cell_type_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'is_primary_data', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'di

## 2. Find the Cell Type Column & List All Cell Types

In [3]:
print("Available metadata columns:")
print(adata.obs.columns.tolist())
print(adata.obs['BICCN_subclass_label'].value_counts())

Available metadata columns:
['aggr_num', 'umi.counts', 'gene.counts', 'library_id', 'tube_barcode', 'Seq_batch', 'Region', 'Lib_type', 'donor_id', 'Amp_Name', 'Amp_Date', 'Amp_PCR_cyles', 'Lib_Date', 'Replicate_Lib', 'Lib_PCR_cycles', 'Lib_PassFail', 'Cell_Capture', 'Lib_Cells', 'Mean_Reads_perCell', 'Median_Genes_perCell', 'Median_UMI_perCell', 'Saturation', 'Live_percent', 'Total_Cells', 'Live_Cells', 'exp_component_name', 'mapped_reads', 'unmapped_reads', 'nonconf_mapped_reads', 'total.reads', 'doublet.score', 'row', 'BICCN_cluster_id', 'QC', 'BICCN_cluster_label', 'BICCN_subclass_label', 'BICCN_class_label', 'size', 'temp_class_label', 'BICCN_ontology_term_id', 'assay_ontology_term_id', 'disease_ontology_term_id', 'tissue_ontology_term_id', 'cell_type_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'is_primary_data', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_

In [4]:
# ── EDIT: set to the column name that contains cell type labels ──────────────
CELLTYPE_COL = "BICCN_subclass_label"
# ─────────────────────────────────────────────────────────────────────────────

cell_types = sorted(adata.obs[CELLTYPE_COL].unique().tolist())
print(f"\nFound {len(cell_types)} cell types in '{CELLTYPE_COL}':\n")
for i, ct in enumerate(cell_types):
    n = (adata.obs[CELLTYPE_COL] == ct).sum()
    print(f"  [{i:02d}] {ct}  (n={n})")


Found 20 cell types in 'BICCN_subclass_label':

  [00] Astro  (n=398)
  [01] Endo  (n=187)
  [02] L2/3 IT  (n=10915)
  [03] L5 ET  (n=161)
  [04] L5 IT  (n=29721)
  [05] L5/6 NP  (n=3147)
  [06] L6 CT  (n=12807)
  [07] L6 IT  (n=4445)
  [08] L6 IT Car3  (n=69)
  [09] L6b  (n=554)
  [10] Lamp5  (n=2357)
  [11] Macrophage  (n=122)
  [12] OPC  (n=146)
  [13] Oligo  (n=537)
  [14] Pvalb  (n=368)
  [15] SMC  (n=11)
  [16] Sncg  (n=348)
  [17] Sst  (n=1869)
  [18] VLMC  (n=55)
  [19] Vip  (n=2966)


## 3. Define Layer V ET and Classify Other Cell Types

In [5]:
# ── EDIT THESE after reading cell type list above ────────────────────────────

LAYER_V_ET = "L5 ET"

NON_NEURON_TYPES = [
    "Astro",
    "Endo",
    "Macrophage",
    "OPC",
    "Oligo",
    "SMC",
    "VLMC",
]

UPPER_LAYER_TYPES = [
    "L2/3 IT",
]

OTHER_DEEP_LAYER_TYPES = [
    "L5 IT",
    "L5/6 NP",
    "L6 CT",
    "L6 IT",
    "L6 IT Car3",
    "L6b",
]

INHIBITORY_TYPES = [
    "Lamp5",
    "Pvalb",
    "Sncg",
    "Sst",
    "Vip",
]

# ── Thresholds ───────────────────────────────────────────────────────────────
LOG2FC_THRESH = 1.0    # absolute log2 fold change cutoff
PVAL_THRESH   = 0.05   # adjusted p-value (FDR)
# ─────────────────────────────────────────────────────────────────────────────

print(f"Target: {LAYER_V_ET}")
print(f"Non-neuron types     : {NON_NEURON_TYPES}")
print(f"Upper layer types    : {UPPER_LAYER_TYPES}")
print(f"Other deep layer     : {OTHER_DEEP_LAYER_TYPES}")
print(f"Inhibitory types     : {INHIBITORY_TYPES}")

Target: L5 ET
Non-neuron types     : ['Astro', 'Endo', 'Macrophage', 'OPC', 'Oligo', 'SMC', 'VLMC']
Upper layer types    : ['L2/3 IT']
Other deep layer     : ['L5 IT', 'L5/6 NP', 'L6 CT', 'L6 IT', 'L6 IT Car3', 'L6b']
Inhibitory types     : ['Lamp5', 'Pvalb', 'Sncg', 'Sst', 'Vip']


## 4. Preprocessing

In [6]:
if adata.raw is not None:
    print("Using adata.raw for DE analysis")
    adata_use = adata.raw.to_adata()
else:
    print("Using adata.X for DE analysis")
    adata_use = adata.copy()

adata_use.obs[CELLTYPE_COL] = adata.obs[CELLTYPE_COL]

if adata_use.X.max() > 100:
    sc.pp.normalize_total(adata_use, target_sum=1e4)
    sc.pp.log1p(adata_use)
    print("Normalized and log1p transformed")
else:
    print("Data appears pre-normalized")

Using adata.raw for DE analysis
Normalized and log1p transformed


## 5. Run DE Analysis: Each Cell Type vs Layer V ET

Same Wilcoxon test as the UP notebook.
`log2FC > 0` = higher in the comparison group = **lower in L5 ET** = `DOWN_in_LayerVET`.

In [7]:
def run_de(adata_use, celltype_col, group_a, group_b, log2fc_thresh, pval_thresh):
    """
    Run Wilcoxon rank-sum DE: group_a vs group_b (Layer V ET).
    log2FC > 0  → higher in group_a → DOWN in Layer V ET
    log2FC < 0  → lower  in group_a → UP   in Layer V ET
    """
    mask = adata_use.obs[celltype_col].isin([group_a, group_b])
    sub  = adata_use[mask].copy()
    sub.obs["group"] = sub.obs[celltype_col].astype(str)

    sc.tl.rank_genes_groups(
        sub,
        groupby="group",
        groups=[group_a],
        reference=group_b,
        method="wilcoxon",
        corr_method="benjamini-hochberg",
        pts=True,
    )

    result = sc.get.rank_genes_groups_df(sub, group=group_a)
    result = result.rename(columns={
        "names"          : "gene",
        "logfoldchanges" : "log2FC",
        "pvals_adj"      : "padj",
        "pvals"          : "pval",
        "scores"         : "score",
    })

    result["-log10padj"] = -np.log10(result["padj"].clip(lower=1e-300))
    result["comparison"] = f"{group_a}_vs_LayerVET"

    sig = result[
        (result["padj"] < pval_thresh) &
        (result["log2FC"].abs() >= log2fc_thresh)
    ].copy()

    # DOWN_in_LayerVET = higher in comparison group (log2FC > 0)
    sig["direction"] = np.where(sig["log2FC"] > 0, "DOWN_in_LayerVET", "UP_in_LayerVET")

    return result, sig


all_comparisons = {}
all_sig         = {}

all_cell_types = NON_NEURON_TYPES + UPPER_LAYER_TYPES + OTHER_DEEP_LAYER_TYPES + INHIBITORY_TYPES

for ct in all_cell_types:
    if ct not in adata_use.obs[CELLTYPE_COL].values:
        print(f"[SKIP] '{ct}' not found in data")
        continue
    print(f"[DE]  {ct} vs {LAYER_V_ET} ...", end=" ")
    full, sig = run_de(adata_use, CELLTYPE_COL, ct, LAYER_V_ET, LOG2FC_THRESH, PVAL_THRESH)
    all_comparisons[ct] = full
    all_sig[ct]         = sig
    n_down = (sig.direction == "DOWN_in_LayerVET").sum()
    n_up   = (sig.direction == "UP_in_LayerVET").sum()
    print(f"{len(sig)} significant genes  (DOWN_in_L5ET={n_down}, UP_in_L5ET={n_up})")

[DE]  Astro vs L5 ET ... 5330 significant genes  (DOWN_in_L5ET=943, UP_in_L5ET=4387)
[DE]  Endo vs L5 ET ... 5527 significant genes  (DOWN_in_L5ET=1166, UP_in_L5ET=4361)
[DE]  Macrophage vs L5 ET ... 5863 significant genes  (DOWN_in_L5ET=1132, UP_in_L5ET=4731)
[DE]  OPC vs L5 ET ... 3903 significant genes  (DOWN_in_L5ET=1020, UP_in_L5ET=2883)
[DE]  Oligo vs L5 ET ... 5182 significant genes  (DOWN_in_L5ET=1198, UP_in_L5ET=3984)
[DE]  SMC vs L5 ET ... 3012 significant genes  (DOWN_in_L5ET=573, UP_in_L5ET=2439)
[DE]  VLMC vs L5 ET ... 4978 significant genes  (DOWN_in_L5ET=586, UP_in_L5ET=4392)
[DE]  L2/3 IT vs L5 ET ... 1837 significant genes  (DOWN_in_L5ET=1170, UP_in_L5ET=667)
[DE]  L5 IT vs L5 ET ... 1541 significant genes  (DOWN_in_L5ET=875, UP_in_L5ET=666)
[DE]  L5/6 NP vs L5 ET ... 1839 significant genes  (DOWN_in_L5ET=911, UP_in_L5ET=928)
[DE]  L6 CT vs L5 ET ... 1483 significant genes  (DOWN_in_L5ET=782, UP_in_L5ET=701)
[DE]  L6 IT vs L5 ET ... 1707 significant genes  (DOWN_in_L5E

## 6. Save Individual Comparison CSVs

In [8]:
for ct, sig in all_sig.items():
    safe_name = ct.replace("/", "_").replace(" ", "_")
    # Save DOWN only — these are the suppressed identity genes
    down_only = sig[sig["direction"] == "DOWN_in_LayerVET"]
    path = OUT_DIR / f"{safe_name}_vs_LayerVET_DOWN.csv"
    down_only.sort_values("log2FC", ascending=False).to_csv(path, index=False)
    print(f"Saved: {path.name}  ({len(down_only)} genes)")

Saved: Astro_vs_LayerVET_DOWN.csv  (943 genes)
Saved: Endo_vs_LayerVET_DOWN.csv  (1166 genes)
Saved: Macrophage_vs_LayerVET_DOWN.csv  (1132 genes)
Saved: OPC_vs_LayerVET_DOWN.csv  (1020 genes)
Saved: Oligo_vs_LayerVET_DOWN.csv  (1198 genes)
Saved: SMC_vs_LayerVET_DOWN.csv  (573 genes)
Saved: VLMC_vs_LayerVET_DOWN.csv  (586 genes)
Saved: L2_3_IT_vs_LayerVET_DOWN.csv  (1170 genes)
Saved: L5_IT_vs_LayerVET_DOWN.csv  (875 genes)
Saved: L5_6_NP_vs_LayerVET_DOWN.csv  (911 genes)
Saved: L6_CT_vs_LayerVET_DOWN.csv  (782 genes)
Saved: L6_IT_vs_LayerVET_DOWN.csv  (1039 genes)
Saved: L6_IT_Car3_vs_LayerVET_DOWN.csv  (905 genes)
Saved: L6b_vs_LayerVET_DOWN.csv  (899 genes)
Saved: Lamp5_vs_LayerVET_DOWN.csv  (1412 genes)
Saved: Pvalb_vs_LayerVET_DOWN.csv  (1426 genes)
Saved: Sncg_vs_LayerVET_DOWN.csv  (1361 genes)
Saved: Sst_vs_LayerVET_DOWN.csv  (1242 genes)
Saved: Vip_vs_LayerVET_DOWN.csv  (1003 genes)


## 7. Stepwise Candidate Narrowing — DOWN direction

At each step we keep genes that are **DOWN in L5 ET** (i.e. higher in the comparison group).
The intersection across all other-deep-layer comparisons at Step 4 ensures the suppression
is truly L5 ET-specific and not just a general deep-layer trend.

In [9]:
def get_layerVET_down_genes(all_sig, cell_types):
    """Genes DOWN in Layer V ET (log2FC > 0, direction == DOWN_in_LayerVET) in ANY comparison."""
    sets = []
    for ct in cell_types:
        if ct in all_sig:
            down = set(all_sig[ct][all_sig[ct]["direction"] == "DOWN_in_LayerVET"]["gene"])
            sets.append(down)
    return set.union(*sets) if sets else set()


def get_layerVET_down_genes_intersect(all_sig, cell_types):
    """Genes DOWN in Layer V ET in ALL comparisons (intersection — stricter)."""
    sets = []
    for ct in cell_types:
        if ct in all_sig:
            down = set(all_sig[ct][all_sig[ct]["direction"] == "DOWN_in_LayerVET"]["gene"])
            sets.append(down)
    return set.intersection(*sets) if sets else set()


# Step 1: DOWN in L5 ET vs any non-neuron → suppressed specifically in neurons
neuron_suppressed_genes = get_layerVET_down_genes(all_sig, NON_NEURON_TYPES)
print(f"Step 1 — Neuron-suppressed genes (DOWN vs non-neurons)         : {len(neuron_suppressed_genes)}")

# Step 2: also DOWN vs inhibitory → suppressed in excitatory neurons specifically
inhibitory_down = get_layerVET_down_genes(all_sig, INHIBITORY_TYPES)
excitatory_suppressed_genes = neuron_suppressed_genes & inhibitory_down
print(f"Step 2 — Excitatory-neuron-suppressed (also DOWN vs inhibitory) : {len(excitatory_suppressed_genes)}")

# Step 3: also DOWN vs upper layer → suppressed in deep layer specifically
upper_layer_down = get_layerVET_down_genes(all_sig, UPPER_LAYER_TYPES)
deep_layer_suppressed_genes = excitatory_suppressed_genes & upper_layer_down
print(f"Step 3 — Deep-layer-suppressed (also DOWN vs upper layer)       : {len(deep_layer_suppressed_genes)}")

# Step 4: DOWN vs ALL other deep layer types (intersection = must be suppressed in every comparison)
consistently_down_vs_deep = get_layerVET_down_genes_intersect(all_sig, OTHER_DEEP_LAYER_TYPES)
layerVET_suppressed_genes = deep_layer_suppressed_genes & consistently_down_vs_deep
print(f"Step 4 — Layer V ET-specific suppression (DOWN vs ALL deep)     : {len(layerVET_suppressed_genes)}")

Step 1 — Neuron-suppressed genes (DOWN vs non-neurons)         : 3864
Step 2 — Excitatory-neuron-suppressed (also DOWN vs inhibitory) : 1171
Step 3 — Deep-layer-suppressed (also DOWN vs upper layer)       : 239
Step 4 — Layer V ET-specific suppression (DOWN vs ALL deep)     : 16


## 8. Build Summary Tables & Save

In [10]:
# ── Gene symbol mapping ───────────────────────────────────────────────────────
print("var columns:", adata.var.columns.tolist())
print(adata.var.head(3))

SYMBOL_COL = None
for candidate in ['feature_name', 'gene_name', 'gene_symbol', 'symbol']:
    if candidate in adata.var.columns:
        SYMBOL_COL = candidate
        break

if SYMBOL_COL:
    id_to_symbol = adata.var[SYMBOL_COL].to_dict()
    print(f"Using '{SYMBOL_COL}' for gene symbols")
else:
    id_to_symbol = {g: g for g in adata.var_names}
    print("No symbol column found — using var_names as gene symbols")

print(f"Loaded {len(id_to_symbol)} gene symbols — example: {list(id_to_symbol.items())[:3]}")

var columns: ['feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type']
                    feature_is_filtered feature_name feature_reference  \
ENSMUSG00000029422                False        Rsrc2   NCBITaxon:10090   
ENSMUSG00000114536                False      Gm48837   NCBITaxon:10090   
ENSMUSG00000049036                False      Tmem121   NCBITaxon:10090   

                   feature_biotype feature_length    feature_type  
ENSMUSG00000029422            gene           1377  protein_coding  
ENSMUSG00000114536            gene           2849          lncRNA  
ENSMUSG00000049036            gene           1574  protein_coding  
Using 'feature_name' for gene symbols
Loaded 30198 gene symbols — example: [('ENSMUSG00000029422', 'Rsrc2'), ('ENSMUSG00000114536', 'Gm48837'), ('ENSMUSG00000049036', 'Tmem121')]


In [12]:
def build_summary(gene_set, all_comparisons, label):
    rows = []
    for gene in sorted(gene_set):
        row = {
            "gene"        : gene,
            "gene_symbol" : id_to_symbol.get(gene, gene),
            "category"    : label,
        }
        for ct, df in all_comparisons.items():
            match = df[df["gene"] == gene]
            if not match.empty:
                row[f"{ct}__log2FC"]    = round(match["log2FC"].values[0], 3)
                row[f"{ct}__log10padj"] = round(match["-log10padj"].values[0], 3)
                row[f"{ct}__padj"]      = match["padj"].values[0]
        rows.append(row)
    return pd.DataFrame(rows)


df_neuron_down    = build_summary(neuron_suppressed_genes,      all_comparisons, "neuron_suppressed")
df_excit_down     = build_summary(excitatory_suppressed_genes,  all_comparisons, "excitatory_neuron_suppressed")
df_deep_down      = build_summary(deep_layer_suppressed_genes,  all_comparisons, "deep_layer_suppressed")
df_specific_down  = build_summary(layerVET_suppressed_genes,    all_comparisons, "LayerVET_specific_suppressed")

df_neuron_down.to_csv(   OUT_DIR / "neuron_suppressed_genes.csv",               index=False)
df_excit_down.to_csv(    OUT_DIR / "excitatory_neuron_suppressed_genes.csv",    index=False)
df_deep_down.to_csv(     OUT_DIR / "deep_layer_suppressed_genes.csv",           index=False)
df_specific_down.to_csv( OUT_DIR / "LayerVET_specific_suppressed_genes.csv",    index=False)

print(f"Saved to {OUT_DIR}/")
print(f"  neuron_suppressed_genes.csv                : {len(df_neuron_down)} genes")
print(f"  excitatory_neuron_suppressed_genes.csv     : {len(df_excit_down)} genes")
print(f"  deep_layer_suppressed_genes.csv            : {len(df_deep_down)} genes")
print(f"  LayerVET_specific_suppressed_genes.csv     : {len(df_specific_down)} genes")

Saved to /home/nakagawa/datasets/LayerV_ET_results_rnaseq/
  neuron_suppressed_genes.csv                : 3864 genes
  excitatory_neuron_suppressed_genes.csv     : 1171 genes
  deep_layer_suppressed_genes.csv            : 239 genes
  LayerVET_specific_suppressed_genes.csv     : 16 genes


## 9. Preview Results

In [13]:
print("\n=== TOP 20 Layer V ET-SPECIFIC SUPPRESSED GENES ===")
print("(log2FC > 0 = higher in comparison group = LOWER in L5 ET)\n")
if not df_specific_down.empty:
    log2fc_cols = [c for c in df_specific_down.columns if c.endswith("__log2FC")]
    df_specific_down["mean_log2FC"] = df_specific_down[log2fc_cols].mean(axis=1)
    display(
        df_specific_down[["gene", "gene_symbol", "mean_log2FC"] + log2fc_cols]
        .sort_values("mean_log2FC", ascending=False)   # highest = most suppressed in L5 ET
        .head(20)
        .reset_index(drop=True)
    )
else:
    print("No genes found — consider relaxing LOG2FC_THRESH / PVAL_THRESH in Cell 3")

print("\n=== TOP 20 DEEP LAYER SUPPRESSED GENES ===")
if not df_deep_down.empty:
    log2fc_cols = [c for c in df_deep_down.columns if c.endswith("__log2FC")]
    df_deep_down["mean_log2FC"] = df_deep_down[log2fc_cols].mean(axis=1)
    display(
        df_deep_down[["gene", "gene_symbol", "mean_log2FC"] + log2fc_cols]
        .sort_values("mean_log2FC", ascending=False)
        .head(20)
        .reset_index(drop=True)
    )


=== TOP 20 Layer V ET-SPECIFIC SUPPRESSED GENES ===
(log2FC > 0 = higher in comparison group = LOWER in L5 ET)



,gene,gene_symbol,mean_log2FC,Astro__log2FC,Endo__log2FC,Macrophage__log2FC,OPC__log2FC,Oligo__log2FC,SMC__log2FC,VLMC__log2FC,...,L5/6 NP__log2FC,L6 CT__log2FC,L6 IT__log2FC,L6 IT Car3__log2FC,L6b__log2FC,Lamp5__log2FC,Pvalb__log2FC,Sncg__log2FC,Sst__log2FC,Vip__log2FC
0,ENSMUSG00000052920,Prkg1,2.668421,-0.227,3.087,0.747,4.742,-0.416,7.370000,6.199000,...,2.509,3.303,3.740,3.084,3.106,0.717,2.540,-0.166,1.852,0.509
1,ENSMUSG00000047045,Tmem164,2.483158,5.154,2.422,4.416,3.047,-1.033,2.287000,3.475000,...,1.268,2.054,1.832,2.122,1.731,2.590,3.186,2.338,3.135,2.459
2,ENSMUSG00000061080,Lsamp,2.460579,5.523,-1.066,-1.335,5.412,0.651,-0.763000,0.705000,...,3.165,2.208,2.201,3.257,2.145,3.024,3.884,3.888,4.534,3.759
3,ENSMUSG00000051359,Ncald,1.919000,-0.867,-0.717,-1.023,3.059,0.898,0.687000,0.804000,...,2.762,3.849,3.317,3.158,3.409,3.201,2.502,1.335,2.506,2.846
4,ENSMUSG00000066705,Fxyd6,1.821842,0.091,-0.162,-0.671,3.057,-0.624,-0.836000,0.545000,...,2.798,3.131,3.573,2.684,2.381,1.789,-0.697,5.933,2.664,4.989
5,ENSMUSG00000043183,Simc1,1.739526,1.191,-0.532,0.835,2.088,1.295,1.404000,-0.747000,...,3.275,1.922,1.758,1.941,1.785,2.666,2.509,2.694,2.647,2.821
6,ENSMUSG00000069662,Marcks,1.704526,-0.750,-0.174,4.574,3.989,0.407,-0.431000,2.707000,...,3.246,1.910,2.167,2.887,2.453,-0.489,-0.893,1.930,2.159,2.177
7,ENSMUSG00000060534,Dcc,1.551368,2.137,-1.576,-1.307,4.516,1.447,-3.133000,-2.015000,...,5.493,4.030,2.376,2.434,2.248,0.709,0.724,1.706,1.961,3.043
8,ENSMUSG00000000686,Abhd15,1.169947,0.264,-0.086,2.497,0.585,0.583,0.430000,-0.869000,...,1.568,1.597,1.709,1.636,1.713,1.898,1.186,1.496,1.366,1.413
9,ENSMUSG00000036469,Marchf1,0.507211,-2.767,-2.364,1.534,4.143,1.465,-2.606000,-2.652000,...,1.556,1.665,1.674,1.951,2.463,-0.936,-0.603,-0.949,0.965,1.817



=== TOP 20 DEEP LAYER SUPPRESSED GENES ===


,gene,gene_symbol,mean_log2FC,Astro__log2FC,Endo__log2FC,Macrophage__log2FC,OPC__log2FC,Oligo__log2FC,SMC__log2FC,VLMC__log2FC,...,L5/6 NP__log2FC,L6 CT__log2FC,L6 IT__log2FC,L6 IT Car3__log2FC,L6b__log2FC,Lamp5__log2FC,Pvalb__log2FC,Sncg__log2FC,Sst__log2FC,Vip__log2FC
0,ENSMUSG00000021709,Erbin,3.448000,4.313,4.564,5.511,4.365,7.197,3.437,4.023,...,2.041,0.974,-0.405,3.094,1.978,4.669,4.522,5.109,3.977,4.742
1,ENSMUSG00000022018,Rgcc,3.032947,6.551,7.321,-0.164,8.678,2.693,4.392,6.166,...,-0.587,0.465,1.963,2.014,-0.270,3.703,5.469,1.818,1.834,0.507
2,ENSMUSG00000020814,Mxra7,2.957105,2.413,6.556,-0.082,2.078,1.411,4.985,5.263,...,-0.789,-0.514,2.649,3.551,-0.737,3.447,4.180,5.351,4.025,5.125
3,ENSMUSG00000034522,Zfp395,2.948105,4.364,4.087,4.347,3.176,3.895,3.892,3.727,...,0.962,2.180,1.892,1.053,2.000,2.293,3.126,3.441,3.738,3.496
4,ENSMUSG00000034055,Phka1,2.736263,7.323,3.628,2.982,5.824,3.591,3.331,3.853,...,0.756,-1.191,-0.627,2.212,0.350,2.988,3.286,3.645,3.019,4.078
5,ENSMUSG00000052920,Prkg1,2.668421,-0.227,3.087,0.747,4.742,-0.416,7.370,6.199,...,2.509,3.303,3.740,3.084,3.106,0.717,2.540,-0.166,1.852,0.509
6,ENSMUSG00000014932,Yes1,2.649947,3.397,5.064,4.042,3.925,3.073,4.007,4.129,...,1.487,-1.649,0.860,2.065,-1.428,3.070,4.349,3.382,3.738,3.943
7,ENSMUSG00000053819,Camk2d,2.647316,2.984,2.146,4.149,2.209,0.632,4.229,3.100,...,4.809,0.297,1.052,1.726,3.234,3.235,2.867,3.828,2.803,3.358
8,ENSMUSG00000021025,Nfkbia,2.625316,3.625,7.288,7.963,2.130,3.467,2.820,4.433,...,1.435,1.024,1.748,1.913,1.607,0.073,1.339,2.098,1.893,1.452
9,ENSMUSG00000050103,Agmo,2.615684,6.314,4.156,8.631,3.803,3.596,2.361,4.349,...,1.505,1.867,1.739,1.478,1.894,1.885,0.202,-0.064,0.826,1.342


## 10. Cross-reference with Epigenetic Discordance Results

Genes here that also appear in `FORCED_SUPPRESSION Tier4` from the discordance notebook
are your **highest-priority candidates**: transcriptionally suppressed AND epigenetically discordant.

In [ ]:
DISCORDANCE_PATH = OUT_DIR / "TOP_PRIORITY_FORCED_SUPPRESSION.csv"

try:
    epi = pd.read_csv(DISCORDANCE_PATH)
    epi_genes = set(epi["gene"])
    print(f"Loaded {len(epi_genes)} FORCED_SUPPRESSION Tier4 genes from discordance notebook")

    overlap = layerVET_suppressed_genes & epi_genes
    print(f"\nOverlap with L5 ET-specific suppressed genes: {len(overlap)}")
    print("These are transcriptionally DOWN + epigenetically discordant — TOP PRIORITY\n")

    if overlap:
        overlap_df = df_specific_down[df_specific_down["gene"].isin(overlap)].copy()
        log2fc_cols = [c for c in overlap_df.columns if c.endswith("__log2FC")]
        overlap_df["mean_log2FC"] = overlap_df[log2fc_cols].mean(axis=1)
        display(
            overlap_df[["gene", "gene_symbol", "mean_log2FC"] + log2fc_cols]
            .sort_values("mean_log2FC", ascending=False)
            .reset_index(drop=True)
        )
        overlap_df.to_csv(OUT_DIR / "TOP_PRIORITY_DOWN_epi_x_rna.csv", index=False)
        print(f"Saved: TOP_PRIORITY_DOWN_epi_x_rna.csv")

except FileNotFoundError:
    print(f"Discordance results not found at {DISCORDANCE_PATH}")
    print("Run the discordance notebook first, or update DISCORDANCE_PATH above")

## 11. Threshold Sensitivity Check (Optional)

In [ ]:
print("Sensitivity check — DOWN gene counts at different thresholds:\n")
print(f"{'log2FC':>8}  {'padj':>6}  {'Neuron':>8}  {'Excit':>8}  {'DeepLayer':>10}  {'L5ET_specific':>14}")

for lfc in [0.5, 1.0, 1.5, 2.0]:
    for pv in [0.1, 0.05, 0.01]:

        def get_down(cts):
            sets = []
            for ct in cts:
                if ct in all_comparisons:
                    df = all_comparisons[ct]
                    # DOWN in L5 ET = log2FC > 0 (higher in comparison group)
                    down = set(df[(df["padj"] < pv) & (df["log2FC"] > lfc)]["gene"])
                    sets.append(down)
            return set.union(*sets) if sets else set()

        def get_down_intersect(cts):
            sets = []
            for ct in cts:
                if ct in all_comparisons:
                    df = all_comparisons[ct]
                    down = set(df[(df["padj"] < pv) & (df["log2FC"] > lfc)]["gene"])
                    sets.append(down)
            return set.intersection(*sets) if sets else set()

        n1 = len(get_down(NON_NEURON_TYPES))
        n2 = len(get_down(NON_NEURON_TYPES) & get_down(INHIBITORY_TYPES))
        n3 = len(get_down(NON_NEURON_TYPES) & get_down(INHIBITORY_TYPES) & get_down(UPPER_LAYER_TYPES))
        n4 = len(get_down(NON_NEURON_TYPES) & get_down(INHIBITORY_TYPES) &
                 get_down(UPPER_LAYER_TYPES) & get_down_intersect(OTHER_DEEP_LAYER_TYPES))
        print(f"{lfc:>8.1f}  {pv:>6.2f}  {n1:>8}  {n2:>8}  {n3:>10}  {n4:>14}")